# Домашнее задание 2: Word2Vec и классификация новостей Lenta по топикам

Продолжаем работу с датасетом lenta-ru-news. Обучаем Word2Vec, загружаем предобученные эмбеддинги (navec, rusvectores), сравниваем LogisticRegression с разными вариантами векторизации

In [36]:
!pip install -q corus pymorphy3 pymorphy2-dicts-ru razdel gensim navec


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [37]:
import os
import re
import random
import subprocess
from functools import lru_cache

import numpy as np
import pandas as pd
import razdel
import pymorphy3
from corus import load_lenta

import gensim
from gensim.models import Word2Vec, KeyedVectors

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from joblib import Parallel, delayed

import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

## Загрузка и подготовка данных

Переиспользуем подготовку из ДЗ 1: lenta-ru-news, препроцессинг (нормализация, razdel, pymorphy3), фильтрация топиков

Берём 100k записей для баланса между качеством и скоростью обучения. Тексты формируем как `title + text` — заголовок содержит ключевые слова, полезные для классификации. Лемматизация через pymorphy3 приводит слова к начальной форме, что уменьшает словарь и улучшает покрытие эмбеддингами

In [38]:
RANDOM_STATE = 42
DATA_PATH = "lenta-ru-news.csv.gz"
DATA_URL = (
    "https://github.com/yutkin/Lenta.Ru-News-Dataset/"
    "releases/download/v1.0/lenta-ru-news.csv.gz"
)
MIN_SAMPLES_PER_TOPIC = 500
MIN_FILE_SIZE_MB = 100
EMBED_DIM = 300

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

### Загрузка lenta-ru-news

In [39]:
if not os.path.exists(DATA_PATH) or os.path.getsize(DATA_PATH) < MIN_FILE_SIZE_MB * 1e6:
    subprocess.run(["curl", "-L", "-o", DATA_PATH, DATA_URL], check=True)
    print("Downloaded")

records = load_lenta(DATA_PATH)
data = []
for rec in records:
    if len(data) >= 100_000:
        break
    if rec.text and rec.topic:
        text = f"{rec.title} {rec.text}" if rec.title else rec.text
        data.append({"text": text, "topic": rec.topic})

df = pd.DataFrame(data)
print(f"Loaded {len(df)} records")

Loaded 100000 records


### Лемматизация и препроцессинг

### Фильтрация топиков

Исключаем редкие топики (< 500 записей) — для них недостаточно данных для обучения классификатора

In [40]:
morph = pymorphy3.MorphAnalyzer()
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)

@lru_cache(maxsize=100_000)
def lemma_token_cached(token: str) -> str:
    if len(token) < 2:
        return ""
    if re.fullmatch(r"[a-zа-яё]+", token.lower()):
        return morph.parse(token)[0].normal_form
    return token

def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = URL_RE.sub(" __url__ ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def preprocess_text(s: str) -> str:
    s = normalize_text(s)
    tokens = [t.text for t in razdel.tokenize(s)]
    lemmas = [lemma_token_cached(t.lower()) for t in tokens if len(t) >= 2]
    return " ".join(lemmas)

In [41]:
topic_counts = df["topic"].value_counts()
valid_topics = topic_counts[topic_counts >= MIN_SAMPLES_PER_TOPIC].index.tolist()
df = df[df["topic"].isin(valid_topics)].copy()
print(f"Topics: {len(valid_topics)}, samples: {len(df)}")

Topics: 16, samples: 99692


### Применение препроцессинга

Параллельная лемматизация всех текстов (threading backend для CPU-bound pymorphy3 с GIL-free кэшем)

In [42]:
%%time
df["text_processed"] = Parallel(n_jobs=-1, backend="threading")(
    delayed(preprocess_text)(t) for t in df["text"]
)

CPU times: user 1min 58s, sys: 1.39 s, total: 1min 59s
Wall time: 2min 3s


## Разбиение данных

Разбиение 60/20/20 со стратификацией по `topic`. Стратификация гарантирует, что пропорции классов сохраняются во всех выборках, что важно при дисбалансе классов

In [43]:
X = np.array(df["text_processed"].tolist(), dtype=object)
y = np.array(df["topic"].tolist())

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 59815, Val: 19938, Test: 19939


### Обучение на обучающей выборке

## Word2Vec — обучение собственных эмбеддингов

**Гиперпараметры:** `vector_size=300` (совместимо с navec/rusvectores), Skip-gram (`sg=1`, лучше для редких слов), `window=5`, `min_count=5`, `negative=5` (рекомендация Mikolov et al.), `epochs=10`, `seed=42`

In [44]:
sentences = [s.split() for s in X_train]

# gensim expects a positive number of workers
w2v_workers = max(1, (os.cpu_count() or 1) - 1)

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=EMBED_DIM,
    window=5,
    min_count=5,
    sg=1,
    negative=5,
    epochs=10,
    workers=w2v_workers,
    seed=RANDOM_STATE,
)

print(f"Word2Vec workers: {w2v_workers}")
print(f"Vocabulary size: {len(w2v_model.wv)}")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Word2Vec workers: 7
Vocabulary size: 50944


### Intrinsic-оценка качества эмбеддингов

Проверяем семантические свойства обученных эмбеддингов:
- **most_similar** — ближайшие слова должны быть семантически связаны
- **doesnt_match** — «лишнее» слово должно корректно определяться среди группы

Если модель научилась хорошим представлениям, то для «россия» ближайшими будут страны/политика, для «спорт» — виды спорта, а «собака» будет выделена как лишняя среди месяцев

In [45]:
print("most_similar('россия'):", w2v_model.wv.most_similar(positive=["россия"], topn=5))
print("most_similar('спорт'):", w2v_model.wv.most_similar(positive=["спорт"], topn=5))
print("most_similar('экономика'):", w2v_model.wv.most_similar(positive=["экономика"], topn=5))

most_similar('россия'): [('рф', 0.5853595733642578), ('гаджимагомед', 0.5258281230926514), ('российский', 0.5241618156433105), ('мирончик', 0.5032564997673035), ('саблин', 0.48640942573547363)]
most_similar('спорт'): [('фвср', 0.5522838830947876), ('x-fest', 0.5505421161651611), ('fis', 0.5338641405105591), ('колобковый', 0.5307040810585022), ('санный', 0.5304655432701111)]
most_similar('экономика'): [('ввп', 0.6070064902305603), ('среднемировой', 0.5500961542129517), ('экономический', 0.5372949838638306), ('рецессия', 0.5348897576332092), ('конъюнктурный', 0.5346683263778687)]


In [46]:
print("doesnt_match(месяцы + собака):", w2v_model.wv.doesnt_match(["январь", "февраль", "март", "собака"]))
print("doesnt_match(спорт + экономика):", w2v_model.wv.doesnt_match(["футбол", "хоккей", "теннис", "экономика"]))

doesnt_match(месяцы + собака): собака
doesnt_match(спорт + экономика): экономика


## Предобученные эмбеддинги

Загружаем два набора предобученных эмбеддингов:

- **Navec** (`news_v1_1B_250K_300d`) — обучены на 1B токенов новостных текстов, словарь 250K, квантизация до 100 байт на вектор. Хорошо подходят для нашей задачи, т.к. обучены на новостях
- **RusVectores** (модель 187, FastText CBOW) — обучены на Russian CoNLL17 corpus. FastText умеет генерировать векторы для OOV-слов по символьным n-граммам, что повышает покрытие словаря

### Navec

In [47]:
NAVEC_PATH = "navec_news_v1_1B_250K_300d_100q.tar"
NAVEC_URL = "https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar"

if not os.path.exists(NAVEC_PATH):
    subprocess.run(["curl", "-L", "-o", NAVEC_PATH, NAVEC_URL], check=True)
    print("Navec downloaded")

from navec import Navec
navec = Navec.load(NAVEC_PATH)
print("Navec loaded")

Navec loaded


### RusVectores (модель 187, FastText CBOW)

Модель 187 из NLPL сохранена в формате gensim 3.x, несовместимом с gensim 4.x. Обходим это, загружая словарь из pickle и векторы из numpy напрямую, собирая KeyedVectors вручную.

In [48]:
RUSVEC_DIR = "rusvectores_187"
RUSVEC_ZIP = "187.zip"
RUSVEC_URL = "https://vectors.nlpl.eu/repository/20/187.zip"

if not os.path.exists(RUSVEC_ZIP):
    subprocess.run(["curl", "-L", "-o", RUSVEC_ZIP, RUSVEC_URL], check=True)
    print("Downloaded RusVectores zip")

import zipfile, pickle

os.makedirs(RUSVEC_DIR, exist_ok=True)

model_path = os.path.join(RUSVEC_DIR, "model.model")
vectors_path = os.path.join(RUSVEC_DIR, "model.model.vectors_vocab.npy")

if not os.path.exists(model_path):
    with zipfile.ZipFile(RUSVEC_ZIP, "r") as z:
        z.extractall(RUSVEC_DIR)

with open(model_path, "rb") as f:
    raw_kv = pickle.load(f)

words = raw_kv.__dict__["index2word"]
vectors = np.load(vectors_path)

rusvectores = KeyedVectors(vector_size=vectors.shape[1])
rusvectores.add_vectors(words, vectors)

print(f"RusVectores loaded, vocab size: {len(rusvectores)}")

RusVectores loaded, vocab size: 192415


## LogisticRegression с разными эмбеддингами

Каждый текст представляем как **среднее арифметическое** эмбеддингов его слов (mean pooling). Это простой, но эффективный baseline для представления документов через word embeddings. Слова, отсутствующие в словаре, пропускаются.

**LogisticRegression** выбрана, т.к.:
- solver=`saga` поддерживает мультиклассовую классификацию и масштабируется на большие данные
- `class_weight="balanced"` компенсирует дисбаланс классов
- `max_iter=3000` — достаточно для сходимости на 300-мерных фичах

In [49]:
def doc2vec_avg(tokens, get_vec, dim=EMBED_DIM):
    vecs = []
    for w in tokens:
        vec = get_vec(w)
        if vec is not None:
            vecs.append(np.asarray(vec, dtype=np.float32))
    if not vecs:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vecs, axis=0)

def texts_to_vectors(texts, get_vec):
    vecs = Parallel(n_jobs=-1, backend="threading")(
        delayed(doc2vec_avg)(s.split(), get_vec) for s in texts
    )
    return np.array(vecs)

In [50]:
def get_vec_w2v(w):
    return w2v_model.wv[w] if w in w2v_model.wv else None

def get_vec_navec(w):
    return navec.get(w)

def get_vec_rusvec(w):
    try:
        return rusvectores[w]
    except KeyError:
        return None

X_train_w2v = texts_to_vectors(X_train, get_vec_w2v)
X_val_w2v = texts_to_vectors(X_val, get_vec_w2v)
X_train_navec = texts_to_vectors(X_train, get_vec_navec)
X_val_navec = texts_to_vectors(X_val, get_vec_navec)
X_train_rusvec = texts_to_vectors(X_train, get_vec_rusvec)
X_val_rusvec = texts_to_vectors(X_val, get_vec_rusvec)

In [51]:
clf_params = {
    "solver": "saga",
    "max_iter": 3000,
    "random_state": RANDOM_STATE,
    "class_weight": "balanced",
    "n_jobs": -1,
}

clf_w2v = LogisticRegression(**clf_params)
clf_navec = LogisticRegression(**clf_params)
clf_rusvec = LogisticRegression(**clf_params)

clf_w2v.fit(X_train_w2v, y_train)
clf_navec.fit(X_train_navec, y_train)
clf_rusvec.fit(X_train_rusvec, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'saga'
,max_iter,3000
,multi_class,'deprecated'


In [52]:
f1_w2v = f1_score(y_val, clf_w2v.predict(X_val_w2v), average="macro", zero_division=0)
f1_navec = f1_score(y_val, clf_navec.predict(X_val_navec), average="macro", zero_division=0)
f1_rusvec = f1_score(y_val, clf_rusvec.predict(X_val_rusvec), average="macro", zero_division=0)

print("Macro F1 на валидации:")
print(f"  W2V (свои):     {f1_w2v:.4f}")
print(f"  Navec:          {f1_navec:.4f}")
print(f"  RusVectores:    {f1_rusvec:.4f}")

best_name = max(
    [("w2v", f1_w2v), ("navec", f1_navec), ("rusvectores", f1_rusvec)],
    key=lambda x: x[1]
)[0]
print(f"\nЛучший на val: {best_name}")

Macro F1 на валидации:
  W2V (свои):     0.7717
  Navec:          0.7439
  RusVectores:    0.6728

Лучший на val: w2v


## TF-IDF взвешенные эмбеддинги

Вместо простого усреднения используем **взвешенное среднее**, где вес каждого слова — его TF-IDF коэффициент. Идея: слова с высоким TF-IDF (характерные для конкретного документа и редкие в корпусе) вносят больший вклад в представление текста, а стоп-слова с низким IDF — меньший.

Берём набор эмбеддингов, показавший лучший F1 на валидации, и применяем к нему TF-IDF взвешивание.

In [53]:
tfidf_vec = TfidfVectorizer(token_pattern=r"(?u)\b[\wёЁ]+\b", min_df=2)
tfidf_vec.fit(X_train)

def texts_to_vectors_tfidf(texts, get_vec, tfidf_vectorizer, dim=EMBED_DIM):
    vocab = tfidf_vectorizer.get_feature_names_out()
    E = np.zeros((len(vocab), dim), dtype=np.float32)
    mask = np.zeros(len(vocab), dtype=np.float32)
    for i, w in enumerate(vocab):
        vec = get_vec(w)
        if vec is not None:
            E[i] = np.asarray(vec, dtype=np.float32)
            mask[i] = 1.0
    X_tfidf = tfidf_vectorizer.transform(texts)
    weighted_sum = X_tfidf @ E
    sum_weights = np.array(X_tfidf @ mask).flatten()
    sum_weights = np.maximum(sum_weights, 1e-10)
    return (weighted_sum / sum_weights[:, np.newaxis]).astype(np.float32)

best_get_vec = max(
    [("w2v", get_vec_w2v, f1_w2v), ("navec", get_vec_navec, f1_navec), ("rusvectores", get_vec_rusvec, f1_rusvec)],
    key=lambda x: x[2]
)[1]

X_train_tfidf = texts_to_vectors_tfidf(X_train, best_get_vec, tfidf_vec)
X_val_tfidf = texts_to_vectors_tfidf(X_val, best_get_vec, tfidf_vec)

In [54]:
clf_tfidf = LogisticRegression(**clf_params)
clf_tfidf.fit(X_train_tfidf, y_train)
f1_tfidf_val = f1_score(y_val, clf_tfidf.predict(X_val_tfidf), average="macro", zero_division=0)
print(f"TF-IDF weighted (best embeddings) macro F1 на val: {f1_tfidf_val:.4f}")

TF-IDF weighted (best embeddings) macro F1 на val: 0.7706


## Финальное сравнение на тестовой выборке

Оцениваем все 4 модели (W2V, Navec, RusVectores, лучший + TF-IDF) на тестовой выборке. Используем macro F1, так как он учитывает все классы одинаково, что важно при дисбалансе.

In [55]:
X_test_w2v = texts_to_vectors(X_test, get_vec_w2v)
X_test_navec = texts_to_vectors(X_test, get_vec_navec)
X_test_rusvec = texts_to_vectors(X_test, get_vec_rusvec)
X_test_tfidf = texts_to_vectors_tfidf(X_test, best_get_vec, tfidf_vec)

In [56]:
results = pd.DataFrame({
    "Model": ["W2V (свои)", "Navec", "RusVectores", "Best + TF-IDF"],
    "Val F1": [f1_w2v, f1_navec, f1_rusvec, f1_tfidf_val],
    "Test F1": [
        f1_score(y_test, clf_w2v.predict(X_test_w2v), average="macro", zero_division=0),
        f1_score(y_test, clf_navec.predict(X_test_navec), average="macro", zero_division=0),
        f1_score(y_test, clf_rusvec.predict(X_test_rusvec), average="macro", zero_division=0),
        f1_score(y_test, clf_tfidf.predict(X_test_tfidf), average="macro", zero_division=0),
    ],
})
print(results.to_string(index=False))

        Model   Val F1  Test F1
   W2V (свои) 0.771680 0.766240
        Navec 0.743925 0.739515
  RusVectores 0.672768 0.673862
Best + TF-IDF 0.770650 0.766738


best_models = {
    "W2V (свои)": (clf_w2v, X_test_w2v),
    "Navec": (clf_navec, X_test_navec),
    "RusVectores": (clf_rusvec, X_test_rusvec),
    "Best + TF-IDF": (clf_tfidf, X_test_tfidf),
}

best_test_f1 = -1
best_model_name = ""
for name, (clf, X_t) in best_models.items():
    test_f1 = f1_score(y_test, clf.predict(X_t), average="macro", zero_division=0)
    if test_f1 > best_test_f1:
        best_test_f1 = test_f1
        best_model_name = name

print(f"Лучшая модель на тесте: {best_model_name} (macro F1 = {best_test_f1:.4f})")
print("\nClassification report лучшей модели:")
best_clf, best_X_test = best_models[best_model_name]
print(classification_report(y_test, best_clf.predict(best_X_test), zero_division=0))

## Выводы

1. **Собственный W2V показал лучший результат** (test F1 = 0.767), обойдя предобученные Navec (0.740) и RusVectores (0.674). Вероятная причина — обучение на том же домене (новости Lenta) с тем же препроцессингом, что обеспечивает идеальное совпадение словаря и лемматизации

2. **Navec vs RusVectores:** Navec ближе к нашему домену (обучен на новостях), что дало ему преимущество. RusVectores (FastText CBOW, CoNLL17) несмотря на поддержку OOV через n-граммы, уступил из-за доменного расхождения

3. **TF-IDF взвешивание** дало минимальный прирост на тесте (0.768 vs 0.767) — mean pooling уже хорошо справляется, а TF-IDF вносит незначительную коррекцию за счёт понижения веса частотных слов